In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install mlflow dagshub -q
import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.environ["MLFLOW_TRACKING_USERNAME"] = "giomamaca"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "095241f97fd50006565a872843e1054bb2ae82cb"
mlflow.set_tracking_uri('https://dagshub.com/giomamaca/ML2.mlflow')
mlflow.set_experiment('RandomForest_Training')
SEED = 42

In [3]:
DAGSHUB_USER = 'giomamaca'
DAGSHUB_REPO = 'ML2'

mlflow.set_tracking_uri(f'https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow')

EXPERIMENT_NAME = 'RandomForest_Training'
mlflow.set_experiment(EXPERIMENT_NAME)

SEED = 42
np.random.seed(SEED)

In [4]:
def run_exists(run_name, experiment_name='RandomForest_Training'):
    client = MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        return False
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'"
    )
    return len(runs) > 0

In [5]:
train_trans  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_trans   = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id      = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = train_trans.merge(train_id, on='TransactionID', how='left')
test  = test_trans.merge(test_id,  on='TransactionID', how='left')

## Preprocessing

In [6]:
class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = X.copy()

        nan_frac        = X.isnull().mean()
        self.nan_drop_  = nan_frac[nan_frac > 0.90].index.tolist()
        num_cols        = [c for c in X.select_dtypes('number').columns
                           if c not in ['TransactionID', 'isFraud']]
        self.near_zero_ = X[num_cols].std()[X[num_cols].std() < 0.01].index.tolist()
        X.drop(columns=self.nan_drop_ + self.near_zero_, inplace=True, errors='ignore')

        X = self._add_features(X)

        self.freq_maps_ = {}
        for col in ['card1', 'card2', 'addr1', 'P_emaildomain']:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(normalize=True).to_dict()

        self.grp_stats_ = {}
        for grp in ['card1', 'card2']:
            if grp in X.columns:
                stats = X.groupby(grp)['TransactionAmt'].agg(['mean', 'std', 'count'])
                stats.columns = [f'{grp}_amt_{s}' for s in ['mean', 'std', 'count']]
                self.grp_stats_[grp] = stats

        if 'card1' in X.columns and 'card2' in X.columns:
            combined = X['card1'].astype(str) + '_' + X['card2'].astype(str)
            self.card_freq_ = combined.value_counts(normalize=True).to_dict()

        self.encoders_ = {}
        for col in X.select_dtypes('object').columns:
            if col not in ['TransactionID']:
                le = LabelEncoder()
                le.fit(X[col].astype(str).fillna('unknown'))
                self.encoders_[col] = le

        print(f'Label encoders fit on {len(self.encoders_)} columns: {list(self.encoders_.keys())}')

        self.feature_names_ = self._transform(X.copy()).columns.tolist()
        return self

    def _add_features(self, X):
        if 'id_30' in X.columns:
            X['os'] = X['id_30'].apply(
                lambda v: next((o for o in ['android','ios','windows','mac','linux']
                                if o in str(v).lower()), 'other') if pd.notna(v) else 'unknown')
        if 'id_31' in X.columns:
            X['browser'] = X['id_31'].apply(
                lambda v: next((b for b in ['chrome','safari','firefox','edge','samsung','opera']
                                if b in str(v).lower()), 'other') if pd.notna(v) else 'unknown')

        if 'TransactionDT' in X.columns:
            X['transaction_day']     = (X['TransactionDT'] // 86400).astype('int32')
            X['transaction_hour']    = ((X['TransactionDT'] // 3600) % 24).astype('int32')
            X['transaction_weekday'] = (X['transaction_day'] % 7).astype('int32')
            X['transaction_month']   = (X['transaction_day'] // 30).astype('int32')
            X['is_night']            = X['transaction_hour'].between(0, 6).astype('int8')
            X['is_weekend']          = X['transaction_weekday'].isin([5, 6]).astype('int8')

        X['TransactionAmt_log']   = np.log1p(X['TransactionAmt'])
        X['TransactionAmt_cents'] = (X['TransactionAmt'] * 100 % 100).astype('float32')
        X['amt_is_round']         = (X['TransactionAmt_cents'] == 0).astype('int8')

        if 'P_emaildomain' in X.columns:
            X['P_email_suffix'] = X['P_emaildomain'].apply(
                lambda x: str(x).split('.')[-1] if pd.notna(x) else 'unknown')
        if 'R_emaildomain' in X.columns:
            X['R_email_suffix'] = X['R_emaildomain'].apply(
                lambda x: str(x).split('.')[-1] if pd.notna(x) else 'unknown')
        if 'P_emaildomain' in X.columns and 'R_emaildomain' in X.columns:
            X['email_match'] = (X['P_emaildomain'] == X['R_emaildomain']).astype('int8')

        return X

    def _transform(self, X):
        for col, fmap in getattr(self, 'freq_maps_', {}).items():
            if col in X.columns:
                X[f'{col}_freq'] = X[col].map(fmap).fillna(0).astype('float32')

        for grp, stats in getattr(self, 'grp_stats_', {}).items():
            if grp in X.columns:
                X = X.merge(stats, on=grp, how='left')

        if hasattr(self, 'card_freq_') and 'card1' in X.columns and 'card2' in X.columns:
            combined = X['card1'].astype(str) + '_' + X['card2'].astype(str)
            X['card1_card2_freq'] = combined.map(self.card_freq_).fillna(0).astype('float32')

        for col, le in getattr(self, 'encoders_', {}).items():
            if col in X.columns:
                X[col] = X[col].astype(str)
                known  = set(le.classes_)
                X[col] = X[col].apply(lambda v: v if v in known else le.classes_[0])
                X[col] = le.transform(X[col])

        drop = [c for c in ['TransactionID', 'TransactionDT', 'isFraud'] if c in X.columns]
        X.drop(columns=drop, inplace=True)

        return X.fillna(-999)

    def transform(self, X):
        X = X.copy()
        drop = [c for c in self.nan_drop_ + self.near_zero_ if c in X.columns]
        X.drop(columns=drop, inplace=True)
        for col in X.select_dtypes('float64').columns:
            X[col] = X[col].astype('float32')
        for col in X.select_dtypes('int64').columns:
            X[col] = X[col].astype('int32')
        X = self._add_features(X)
        X = self._transform(X)

        for col in self.feature_names_:
            if col not in X.columns:
                X[col] = -999
        return X[self.feature_names_].fillna(-999)

## Cleaning + Feature Engineering

In [7]:
preprocessor = FraudPreprocessor()

if not run_exists('RandomForest_Cleaning'):
    with mlflow.start_run(run_name='RandomForest_Cleaning'):
        preprocessor.fit(train.drop(columns=['isFraud']))
        mlflow.log_params({
            'dropped_high_nan' : len(preprocessor.nan_drop_),
            'dropped_near_zero': len(preprocessor.near_zero_),
            'remaining_cols'   : len(preprocessor.feature_names_)
        })
else:
    print('RandomForest_Cleaning already logged, skipping...')
    preprocessor.fit(train.drop(columns=['isFraud']))

if not run_exists('RandomForest_Feature_Engineering'):
    with mlflow.start_run(run_name='RandomForest_Feature_Engineering'):
        mlflow.log_metric('total_features', len(preprocessor.feature_names_))
else:
    print('RandomForest_Feature_Engineering already logged, skipping...')

X      = preprocessor.transform(train.drop(columns=['isFraud']))
X_test = preprocessor.transform(test)
y      = train['isFraud']

print('X shape     :', X.shape)
print('X_test shape:', X_test.shape)

RandomForest_Cleaning already logged, skipping...
Label encoders fit on 33 columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo', 'os', 'browser', 'P_email_suffix', 'R_email_suffix']
RandomForest_Feature_Engineering already logged, skipping...


/tmp/ipykernel_977/845200833.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['os'] = X['id_30'].apply(
/tmp/ipykernel_977/845200833.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['browser'] = X['id_31'].apply(
/tmp/ipykernel_977/845200833.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy

X shape     : (590540, 442)
X_test shape: (506691, 442)


## Feature Selection

In [8]:
TARGET    = 'isFraud'
DROP_COLS = ['TransactionID', 'TransactionDT', TARGET]

X = preprocessor.transform(train.drop(columns=['isFraud']))
y = train[TARGET]

for col in X.select_dtypes('object').columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str).fillna('unknown'))

/tmp/ipykernel_977/845200833.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['os'] = X['id_30'].apply(
/tmp/ipykernel_977/845200833.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['browser'] = X['id_31'].apply(
/tmp/ipykernel_977/845200833.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy

In [9]:
SKF_FAST = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
SKF      = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

BASE_PARAMS = dict(
    n_estimators=200, max_depth=16,
    class_weight='balanced',
    random_state=SEED, n_jobs=-1
)

sample_idx = X.sample(frac=0.3, random_state=SEED).index
X_sample   = X.loc[sample_idx]
y_sample   = y.loc[sample_idx]

if not run_exists('RandomForest_Feature_Selection'):
    with mlflow.start_run(run_name='RandomForest_Feature_Selection'):

        num_X     = X.select_dtypes(include='number')
        sample    = num_X.sample(frac=0.2, random_state=SEED)
        corr      = sample.corr().abs()
        upper     = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
        feats_A   = [c for c in X.columns if c not in drop_corr]
        score_A   = cross_val_score(RandomForestClassifier(**BASE_PARAMS),
                                    X_sample[feats_A], y_sample,
                                    cv=SKF_FAST, scoring='roc_auc').mean()
        with mlflow.start_run(run_name='RF_FS_A_corr', nested=True):
            m_A = RandomForestClassifier(**BASE_PARAMS)
            m_A.fit(X[feats_A], y)
            mlflow.log_metric('cv_auc', score_A)
            mlflow.log_param('n_features', len(feats_A))
            mlflow.sklearn.log_model(m_A, artifact_path='model',
                                     registered_model_name='RF_FS_A_corr')
        print(f'A done: {score_A:.5f}')

        rf_imp      = RandomForestClassifier(**BASE_PARAMS)
        rf_imp.fit(X, y)
        importances = pd.Series(rf_imp.feature_importances_, index=X.columns)
        feats_B     = importances.nlargest(150).index.tolist()
        score_B     = cross_val_score(RandomForestClassifier(**BASE_PARAMS),
                                      X_sample[feats_B], y_sample,
                                      cv=SKF_FAST, scoring='roc_auc').mean()
        with mlflow.start_run(run_name='RF_FS_B_top150', nested=True):
            m_B = RandomForestClassifier(**BASE_PARAMS)
            m_B.fit(X[feats_B], y)
            mlflow.log_metric('cv_auc', score_B)
            mlflow.log_param('n_features', len(feats_B))
            mlflow.sklearn.log_model(m_B, artifact_path='model',
                                     registered_model_name='RF_FS_B_top150')
        print(f'B done: {score_B:.5f}')

        feats_C = [f for f, keep in zip(X.columns,
                   SelectFromModel(rf_imp, threshold='median', prefit=True).get_support()) if keep]
        score_C = cross_val_score(RandomForestClassifier(**BASE_PARAMS),
                                  X_sample[feats_C], y_sample,
                                  cv=SKF_FAST, scoring='roc_auc').mean()
        with mlflow.start_run(run_name='RF_FS_C_sfm', nested=True):
            m_C = RandomForestClassifier(**BASE_PARAMS)
            m_C.fit(X[feats_C], y)
            mlflow.log_metric('cv_auc', score_C)
            mlflow.log_param('n_features', len(feats_C))
            mlflow.sklearn.log_model(m_C, artifact_path='model',
                                     registered_model_name='RF_FS_C_sfm')
        print(f'C done: {score_C:.5f}')

        rfe = RFE(RandomForestClassifier(n_estimators=50, max_depth=10,
                                          random_state=SEED, n_jobs=-1),
                  n_features_to_select=100, step=20)
        rfe.fit(X_sample, y_sample)
        feats_D = X.columns[rfe.support_].tolist()
        score_D = cross_val_score(RandomForestClassifier(**BASE_PARAMS),
                                  X_sample[feats_D], y_sample,
                                  cv=SKF_FAST, scoring='roc_auc').mean()
        with mlflow.start_run(run_name='RF_FS_D_rfe', nested=True):
            m_D = RandomForestClassifier(**BASE_PARAMS)
            m_D.fit(X[feats_D], y)
            mlflow.log_metric('cv_auc', score_D)
            mlflow.log_param('n_features', len(feats_D))
            mlflow.sklearn.log_model(m_D, artifact_path='model',
                                     registered_model_name='RF_FS_D_rfe')
        print(f'D done: {score_D:.5f}')

        BEST_FS_NAME, BEST_FS_SCORE, SELECTED_FEATURES = max(
            [('A_corr',   score_A, feats_A),
             ('B_top150', score_B, feats_B),
             ('C_sfm',    score_C, feats_C),
             ('D_rfe',    score_D, feats_D)],
            key=lambda x: x[1]
        )

        mlflow.log_metrics({'score_A': score_A, 'score_B': score_B,
                            'score_C': score_C, 'score_D': score_D,
                            'best_score': BEST_FS_SCORE})
        mlflow.log_param('best_approach', BEST_FS_NAME)
        print(f'Best: {BEST_FS_NAME} — {BEST_FS_SCORE:.5f} ({len(SELECTED_FEATURES)} features)')

else:
    print('RandomForest_Feature_Selection already logged, recovering from MLflow...')
    client = MlflowClient()
    exp    = client.get_experiment_by_name('RandomForest_Training')
    runs   = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="tags.mlflow.runName = 'RandomForest_Feature_Selection'"
    )
    run         = runs[0]
    BEST_FS_NAME = run.data.params['best_approach']
    print(f'Best approach was: {BEST_FS_NAME}')

    num_X     = X.select_dtypes(include='number')
    sample    = num_X.sample(frac=0.2, random_state=SEED)
    corr      = sample.corr().abs()
    upper     = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
    feats_A   = [c for c in X.columns if c not in drop_corr]

    rf_imp      = RandomForestClassifier(**BASE_PARAMS)
    rf_imp.fit(X, y)
    importances = pd.Series(rf_imp.feature_importances_, index=X.columns)
    feats_B     = importances.nlargest(150).index.tolist()
    feats_C     = [f for f, keep in zip(X.columns,
                   SelectFromModel(rf_imp, threshold='median', prefit=True).get_support()) if keep]
    rfe = RFE(RandomForestClassifier(n_estimators=50, max_depth=10,
                                      random_state=SEED, n_jobs=-1),
              n_features_to_select=100, step=20)
    rfe.fit(X_sample, y_sample)
    feats_D = X.columns[rfe.support_].tolist()

    SELECTED_FEATURES = {'A_corr': feats_A, 'B_top150': feats_B,
                         'C_sfm': feats_C, 'D_rfe': feats_D}[BEST_FS_NAME]
    print(f'SELECTED_FEATURES recovered: {len(SELECTED_FEATURES)} features')

RandomForest_Feature_Selection already logged, recovering from MLflow...
Best approach was: D_rfe
SELECTED_FEATURES recovered: 100 features


## Training

In [10]:
X_sel  = X[SELECTED_FEATURES]
X_test = X_test[SELECTED_FEATURES]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_sel, y, test_size=0.2, random_state=SEED, stratify=y
)

def get_state(train_auc, val_auc):
    gap = train_auc - val_auc
    if val_auc < 0.80 and gap < 0.02:
        return 'underfitting'
    elif gap > 0.05:
        return 'overfitting'
    else:
        return 'good'

In [11]:
def run_experiment(run_name, params, X_tr, y_tr, X_val, y_val):
    if run_exists(run_name):
        print(f'{run_name} already logged, skipping...')
        return
    with mlflow.start_run(run_name=run_name):
        m = RandomForestClassifier(**params)
        m.fit(X_tr, y_tr)
        train_auc = roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        state     = get_state(train_auc, val_auc)
        print(f'{run_name}: Train={train_auc:.5f}  Val={val_auc:.5f}  → {state}')
        mlflow.log_params(params)
        mlflow.log_metrics({'train_auc': train_auc, 'val_auc': val_auc,
                            'gap': train_auc - val_auc})
        mlflow.log_param('model_state', state)

In [12]:
run_experiment(
    run_name = 'RandomForest_depth3_trees10',
    params   = dict(n_estimators=10, max_depth=3,
                    random_state=SEED, n_jobs=-1),
    X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val
)

RandomForest_depth3_trees10 already logged, skipping...


In [13]:
run_experiment(
    run_name = 'RandomForest_depthNone_noreg',
    params   = dict(n_estimators=500, max_depth=None,
                    min_samples_leaf=1, min_samples_split=2,
                    random_state=SEED, n_jobs=-1),
    X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val
)

RandomForest_depthNone_noreg already logged, skipping...


In [14]:
run_experiment(
    run_name = 'RandomForest_depth16_baseline',
    params   = dict(n_estimators=300, max_depth=16,
                    min_samples_leaf=4, min_samples_split=10,
                    class_weight='balanced',
                    random_state=SEED, n_jobs=-1),
    X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val
)

RandomForest_depth16_baseline already logged, skipping...


In [18]:
def objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 100, 800, step=100),
        max_depth         = trial.suggest_int('max_depth', 8, 32),
        min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 20),
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20),
        max_features      = trial.suggest_float('max_features', 0.1, 1.0),
        class_weight      = 'balanced',
        random_state      = SEED,
        n_jobs            = -1
    )
    m = RandomForestClassifier(**params)
    m.fit(X_tr, y_tr)
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
    with mlflow.start_run(run_name=f'RF_trial_{trial.number}', nested=True):
        mlflow.log_params(params)
        mlflow.log_metric('val_auc', val_auc)
    return val_auc

if not run_exists('RandomForest_HyperparamSearch_v2'):
    with mlflow.start_run(run_name='RandomForest_HyperparamSearch'):
        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective, n_trials=5, show_progress_bar=True)
        best_params = study.best_params
        best_score  = study.best_value
        print(f'Best Val AUC: {best_score:.5f}')
        mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
        mlflow.log_metric('best_val_auc', best_score)
else:
    print('RandomForest_HyperparamSearch already logged, recovering best_params...')
    client = MlflowClient()
    exp    = client.get_experiment_by_name('RandomForest_Training')
    runs   = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="tags.mlflow.runName = 'RandomForest_HyperparamSearch'"
    )
    run = runs[0]
    best_params = {
        'n_estimators'     : int(run.data.params['best_n_estimators']),
        'max_depth'        : int(run.data.params['best_max_depth']),
        'min_samples_leaf' : int(run.data.params['best_min_samples_leaf']),
        'min_samples_split': int(run.data.params['best_min_samples_split']),
        'max_features'     : float(run.data.params['best_max_features']),
    }
    print('best_params recovered:', best_params)

  0%|          | 0/5 [00:00<?, ?it/s]

🏃 View run RF_trial_0 at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/1/runs/0ccb21fb5b204f538f88a58551de1e40
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/1
[W 2026-05-07 14:12:51,332] Trial 1 failed with parameters: {'n_estimators': 200, 'max_depth': 9, 'min_samples_leaf': 18, 'min_samples_split': 13, 'max_features': 0.737265320016441} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_977/4019287833.py", line 13, in objective
    m.fit(X_tr, y_tr)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_forest.py", line 487,

KeyboardInterrupt: 

In [16]:
print(run.data.params)

{'best_approach': 'D_rfe'}


In [20]:
if not run_exists('RandomForest_Final_Model_v2'):
    with mlflow.start_run(run_name='RandomForest_Final_Model_v2'):
        FINAL_PARAMS = {
            **best_params,
            'class_weight': 'balanced',
            'random_state': SEED,
            'n_jobs'      : -1
        }

        oof = np.zeros(len(y))
        for fold, (tr_idx, val_idx) in enumerate(SKF.split(X_sel, y), 1):
            m = RandomForestClassifier(**FINAL_PARAMS)
            m.fit(X_sel.iloc[tr_idx], y.iloc[tr_idx])
            oof[val_idx] = m.predict_proba(X_sel.iloc[val_idx])[:, 1]

        final_auc = roc_auc_score(y, oof)
        print(f'Final OOF AUC: {final_auc:.5f}')

        preprocessor.feature_names_ = SELECTED_FEATURES

        final_clf = RandomForestClassifier(**FINAL_PARAMS)
        final_clf.fit(X_sel, y)

        fraud_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier',   final_clf)
        ])

        mlflow.log_metric('final_oof_auc', final_auc)
        mlflow.sklearn.log_model(
            fraud_pipeline,
            artifact_path='randomforest_pipeline',
            registered_model_name='RandomForest_Fraud_Pipeline'
        )
else:
    print('RandomForest_Final_Model_v2 already logged, skipping...')

Final OOF AUC: 0.94898


2026/05/07 14:56:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 14:56:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'RandomForest_Fraud_Pipeline'.
2026/05/07 14:57:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForest_Fraud_Pipeline, version 1
Created version '1' of model 'RandomForest_Fraud_Pipeline'.


🏃 View run RandomForest_Final_Model_v2 at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/1/runs/aa23e1a5b96a4389997925925ca54c44
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/1
